# Notebook 03: Text Preprocessing

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
Text preprocessing transforms unstructured raw text into clean, standardized tokens suitable for Information Retrieval indexing (Inverted Index, TF-IDF, BM25). 

In Phnom Penh, restaurant documents are **bilingual**, containing both English and Khmer text. This notebook demonstrates our configurable, language-aware `TextPreprocessor`.

**Core Objectives:**
1. Demonstrate lowercase normalization and punctuation removal.
2. Showcase English tokenization and stopword removal.
3. Showcase Khmer word segmentation (via `khmer-nltk`) without applying English stopwords to Khmer text.
4. Preprocess the entire 698-restaurant corpus and inspect vocabulary statistics.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor

print(f"Project root: {PROJECT_ROOT}")

## 1. Initialize the Configurable TextPreprocessor

We instantiate `TextPreprocessor` with standard IR settings.

In [ ]:
preprocessor = TextPreprocessor(
    lowercase=True,
    remove_punctuation=True,
    remove_stopwords=True,
    enable_khmer=True,
    min_token_length=1
)
print("Preprocessor initialized successfully!")

## 2. Test English Text Preprocessing

Observe lowercase conversion, punctuation removal, and stopword filtering (`the`, `in`, `and` removed).

In [ ]:
sample_english = "The Best Authentic Italian Pizza & Pasta in Boeung Keng Kang 1 (BKK1), Phnom Penh!"
tokens_english = preprocessor.preprocess(sample_english)

print("Raw Text: ", sample_english)
print("Tokens:   ", tokens_english)

## 3. Test Khmer Word Segmentation

Khmer script does not use spaces between words. The preprocessor uses `khmer-nltk` to segment continuous text into semantic words without stripping English stopwords.

In [ ]:
sample_khmer = "ហាងមី ឆាយ អ៊ីម ផ្លូវភ្លោះអូឫស្សី"
tokens_khmer = preprocessor.preprocess(sample_khmer)

print("Raw Khmer Text: ", sample_khmer)
print("Segmented Tokens:", tokens_khmer)

## 4. Test Bilingual Text Preprocessing

Real restaurant documents frequently combine English and Khmer in the same name or address.

In [ ]:
sample_bilingual = "Bai Sor បាយស | Traditional Khmer Food at St 240, Daun Penh, ភ្នំពេញ"
tokens_bilingual = preprocessor.preprocess(sample_bilingual)

print("Raw Bilingual: ", sample_bilingual)
print("Tokens:        ", tokens_bilingual)

## 5. Batch Preprocess the Restaurant Corpus

We now load all 698 restaurants, convert them to documents, and preprocess the entire corpus.

In [ ]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)
corpus = converter.get_corpus()

# Preprocess all documents
tokenized_corpus = preprocessor.preprocess_documents(corpus)
print(f"Preprocessed {len(tokenized_corpus)} documents.")

# Inspect first 3 tokenized documents
for doc_id in list(tokenized_corpus.keys())[:3]:
    print(f"\n[Doc {doc_id}]: {tokenized_corpus[doc_id]}")

## 6. Corpus Preprocessing Analytics

Let's inspect vocabulary size and token frequency distribution.

In [ ]:
from collections import Counter

all_tokens = [token for doc_tokens in tokenized_corpus.values() for token in doc_tokens]
token_counts = Counter(all_tokens)

print(f"Total Tokens across corpus : {len(all_tokens):,}")
print(f"Unique Vocabulary Size     : {len(token_counts):,}")

print("\nTop 20 Most Frequent Terms:")
for term, count in token_counts.most_common(20):
    print(f"  {term:20s}: {count:4d}")

## 7. Observations
- Dual-language tokenization cleanly segments both English words and unspaced Khmer terms.
- English stopwords are eliminated while authentic Khmer terms (*មី, បាយ, ហាង, ភ្នំពេញ*) are fully preserved.
- Output `tokenized_corpus` is ready to be ingested by `InvertedIndex` in `04_inverted_index.ipynb`.